# Test a branch with the real model, before publishing

Runs the app from a GitHub branch in Colab, with the real embedding model and (optionally) your Groq key:

1. Downloads the branch
2. Installs the same packages the Hugging Face Space uses
3. Runs the automated tests
4. Runs a few real searches and one AI analysis
5. Starts the app with a temporary public link you can click through

**Groq key (optional):** click the key icon in Colab's left sidebar → *Add new secret* → name `GROQ_API_KEY`, paste your key, and switch on *Notebook access*. Without it, everything works except the AI analysis.

Run the cells from top to bottom (or *Runtime → Run all*). A CPU runtime is enough.

In [ ]:
BRANCH = "v2-restructure"   # the branch to test

!rm -rf /content/app
!git clone -q --depth 1 -b {BRANCH} https://github.com/Renuk9390/patent-prior-art-explorer.git /content/app
%cd /content/app
!git log --oneline -1

In [ ]:
# Same Gradio version as the Space (sdk_version in README.md), plus the app's packages
import re
sdk = re.search(r'sdk_version:\s*"?([\d.]+)', open("README.md").read()).group(1)
print("Gradio version used by the Space:", sdk)
!pip install -q "gradio=={sdk}" -r requirements.txt pytest

import importlib.metadata as md
for pkg in ["gradio", "sentence-transformers", "torch", "faiss-cpu", "numpy", "pypdf", "groq", "pyyaml"]:
    try:
        print(f"{pkg:22} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:22} not installed")

In [ ]:
# Automated tests (fake model, a few seconds)
!python -m pytest -q tests

In [ ]:
import os, sys
sys.path[:0] = ["/content/app/src", "/content/app"]

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("Groq key loaded from Colab secrets.")
except Exception as exc:
    print("No Groq key (AI analysis will show the 'unavailable' notice):", type(exc).__name__)

In [ ]:
# Real searches with the real embedding model
from patentsearch.service import SearchService

service = SearchService.from_config()
for query in ["preventing a battery from overheating",
              "wireless charging of a mobile device",
              "drug delivery through the skin without needles"]:
    print(f"\n=== {query}")
    for r in service.search(query):
        print(f"  {r.score:.3f}  {r.chunk.doc_id:22} {r.chunk.doc_title}")
    best = service.search(query)[0]
    print("  highlighted:", service.explain(query, best.chunk.text)["highlighted"][0][:110], "...")

In [ ]:
# One real AI analysis (needs the Groq key)
query = "preventing a battery from overheating"
best = service.search(query)[0]
text = service.analyse(query, best.chunk.text)
print(text if text else "No analysis returned (no key, or Groq unavailable).")

In [ ]:
# Start the app with a temporary public link (stop this cell when done)
import apps.gradio_app as ui
ui._SERVICE = service   # reuse the already-loaded model
ui.build_app().launch(share=True, css=ui.CUSTOM_CSS)